In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to Asia/Shanghai timezone-aware ISO strings (handle empty)
def ts_range_iso(s):
    if s.empty:
        return (None, None)
    ts = pd.to_datetime(s, unit='s')
    # treat timestamps as local to Asia/Shanghai per instructions
    ts = ts.dt.tz_localize(tz)
    return (ts.min().isoformat(), ts.max().isoformat())

# Summaries for each file
summaries = []

# metric.csv summary
min_ts, max_ts = ts_range_iso(metric_df['timestamp']) if 'timestamp' in metric_df.columns else (None, None)
metric_top_names = metric_df['kpi_name'].value_counts().head(20)
metric_top_list = list(metric_top_names.items())
metric_summary = {
    'file_name': 'metric.csv',
    'row_count': len(metric_df),
    'min_timestamp': min_ts,
    'max_timestamp': max_ts,
    'sorted_unique_cmdb_ids': sorted(metric_df['cmdb_id'].dropna().unique()),
    'top_20_unique_names_with_counts': metric_top_list
}
summaries.append(metric_summary)

# trace.csv summary
min_ts, max_ts = ts_range_iso(trace_df['timestamp']) if 'timestamp' in trace_df.columns else (None, None)
trace_top = trace_df['trace_name'].value_counts().head(20)
trace_summary = {
    'file_name': 'trace.csv',
    'row_count': len(trace_df),
    'min_timestamp': min_ts,
    'max_timestamp': max_ts,
    'sorted_unique_cmdb_ids': sorted(trace_df['cmdb_id'].dropna().unique()),
    'top_20_unique_names_with_counts': list(trace_top.items())
}
summaries.append(trace_summary)

# log.csv summary
min_ts, max_ts = ts_range_iso(log_df['timestamp']) if 'timestamp' in log_df.columns else (None, None)
log_top = log_df['log_name'].value_counts().head(20)
log_summary = {
    'file_name': 'log.csv',
    'row_count': len(log_df),
    'min_timestamp': min_ts,
    'max_timestamp': max_ts,
    'sorted_unique_cmdb_ids': sorted(log_df['cmdb_id'].dropna().unique()),
    'top_20_unique_names_with_counts': list(log_top.items())
}
summaries.append(log_summary)

# error_logs.csv summary - use first 50 chars of message as prefix
min_ts, max_ts = ts_range_iso(error_df['timestamp']) if 'timestamp' in error_df.columns else (None, None)
if 'message' in error_df.columns:
    prefixes = error_df['message'].fillna('').astype(str).str[:50]
    prefix_counts = prefixes.value_counts().head(20)
    prefix_list = list(prefix_counts.items())
else:
    prefix_list = []
error_summary = {
    'file_name': 'error_logs.csv',
    'row_count': len(error_df),
    'min_timestamp': min_ts,
    'max_timestamp': max_ts,
    'sorted_unique_cmdb_ids': sorted(error_df['cmdb_id'].dropna().unique()) if 'cmdb_id' in error_df.columns else [],
    'top_20_unique_names_with_counts': prefix_list
}
summaries.append(error_summary)

summary_df = pd.DataFrame(summaries)

# Additional metric aggregation: by (cmdb_id, kpi_name) compute count, mean, p50, p95, p99
if not metric_df.empty:
    # Ensure value column numeric
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    agg = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99)
    ).reset_index()
    metric_agg_top50 = agg.sort_values('count', ascending=False).head(50)
else:
    metric_agg_top50 = pd.DataFrame(columns=['cmdb_id','kpi_name','count','mean','p50','p95','p99'])

# Display compact outputs: the summary dataframe (one row per file) and top 50 metric aggregates
summary_df, metric_agg_top50

```
Out[1]:
```
summary_text = (
    "Compact summary:\n"
    "- metric.csv: 27,666 rows; time range 2021-03-06T17:30:00+08:00 to 2021-03-06T18:00:00+08:00. "
    "CMDB ids include IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01/02, Tomcat01-04, ... . "
    "Top metric names (examples) include OSLinux network/process metrics and many JVM/Tomcat KPIs (top 20 shown in results).\n"
    "- trace.csv: 6,474 rows; time range 2021-03-06T17:30:00+08:00 to 2021-03-06T17:59:00+08:00. "
    "CMDB ids similar to metric file. Top trace feature: trace.self.duration_mean (360 occurrences) among others.\n"
    "- log.csv: 384 rows; time range 2021-03-06T17:30:00+08:00 to 2021-03-06T18:00:00+08:00. "
    "CMDB ids include Mysql01 and Tomcat nodes. Top log names: log.error_count (186), log.row_count (186), etc.\n"
    "- error_logs.csv: 0 rows (no error messages present).\n\n"
    "Metric aggregation (by cmdb_id, kpi_name): top 50 groups sorted by count were returned. "
    "Many top groups have count = 31 (full-minute coverage for the period). Examples of top aggregated rows:\n"
    "  - IG01 / JVM-Threads_7778_JVM_ThreadCount_Threads — count 31, mean ~69, p50 69.0, p95 ~75.7\n"
    "  - IG01 / JVM-Memory_NoHeapMemoryUsed — count 31, mean ~57,959,730\n"
    "  - IG01 / JVM-Memory_HeapMemoryUsed — count 31, mean ~709,546,100, p50 722,109,856\n"
    "  - Mysql01 / various OSLinux process metrics — count 31, many zeros or small values\n"
    "  - Multiple Tomcat03 request/thread/session KPIs appear among top groups (count 31).\n\n"
    "If you want more detail for any file (e.g., full list of top metric names or more aggregated rows), ask for df.head(X) for that output."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(        file_name  row_count              min_timestamp              max_timestamp                             sorted_unique_cmdb_ids                    top_20_unique_names_with_counts
0      metric.csv      27666  2021-03-06T17:30:00+08:00  2021-03-06T18:00:00+08:00  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...  [(OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT...
1       trace.csv       6474  2021-03-06T17:30:00+08:00  2021-03-06T17:59:00+08:00  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  [(trace.self.duration_mean, 360), (trace.self....
2         log.csv        384  2021-03-06T17:30:00+08:00  2021-03-06T18:00:00+08:00  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...  [(log.error_count, 186), (log.row_count, 186),...
3  error_logs.csv          0                       None                       None                                                 []                                                 [],        cmdb_id                                           kpi_name  count          mean          p50          p95          p99
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.954839e+01         69.0         75.0         75.7
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.795973e+07   57958440.0   57962088.0   57962088.0
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  7.095461e+08  722109856.0  855647536.0  867794917.6
577    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00          0.0          0.0          0.0
580    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  3.967742e-01          0.4          0.4          0.4
...        ...                                                ...    ...           ...          ...          ...          ...
1391  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  5.343073e+05     534329.0     534515.5     534529.4
1405  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  6.451613e-02          0.0          0.5          1.0
1406  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02        500.0        500.0        500.0
1392  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00          0.0          0.0          0.0
1402  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00          0.0          0.0          0.0

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse tz and dataframes from prior state
tz = pytz.timezone('Asia/Shanghai')

# define window boundaries in Asia/Shanghai (UTC+8)
window_start = pd.Timestamp('2021-03-07 01:30:00', tz=tz)
window_end = pd.Timestamp('2021-03-07 02:00:00', tz=tz)

# helper to get localized timestamps Series from an epoch-second column
def localized_ts_series(df):
    return pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

# helper to build row for each file
def file_window_summary(df, file_name, name_limit=20):
    if df.empty:
        rows_in_window = 0
        min_ts = None
        max_ts = None
        cmdbs = []
    else:
        ts = localized_ts_series(df)
        mask = (ts >= window_start) & (ts <= window_end)
        sub = df[mask].copy()
        rows_in_window = int(sub.shape[0])
        if rows_in_window == 0:
            min_ts = None
            max_ts = None
            cmdbs = []
        else:
            ts_sub = pd.to_datetime(sub['timestamp'], unit='s').dt.tz_localize(tz)
            min_ts = ts_sub.min().isoformat()
            max_ts = ts_sub.max().isoformat()
            cmdbs = sorted(sub['cmdb_id'].dropna().unique())[:name_limit]
    return {
        'file_name': file_name,
        'rows_in_window': rows_in_window,
        'min_timestamp_in_window': min_ts,
        'max_timestamp_in_window': max_ts,
        'sorted_unique_cmdb_ids_in_window': cmdbs
    }

# build summary rows for each file
summary_rows = []
summary_rows.append(file_window_summary(metric_df, 'metric.csv'))
summary_rows.append(file_window_summary(trace_df, 'trace.csv'))
summary_rows.append(file_window_summary(log_df, 'log.csv'))
summary_rows.append(file_window_summary(error_df, 'error_logs.csv'))

window_summary_df = pd.DataFrame(summary_rows)

# For metric.csv: top 20 (cmdb_id, kpi_name) groups by row count inside the window
ts_metric = localized_ts_series(metric_df)
mask_metric = (ts_metric >= window_start) & (ts_metric <= window_end)
metric_in_window = metric_df[mask_metric].copy()

if metric_in_window.empty:
    metric_top20_in_window = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'count'])
else:
    metric_top20_in_window = (
        metric_in_window
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp']
        .count()
        .rename(columns={'timestamp': 'count'})
        .sort_values('count', ascending=False)
        .head(20)
    )

# display compact summaries
window_summary_df, metric_top20_in_window

```
Out[2]:
```
summary_text = (
    "Summary for window 2021-03-07 01:30:00 to 2021-03-07 02:00:00 (UTC+8):\n"
    "- metric.csv: 0 rows in window; no timestamps or CMDB ids present in this window.\n"
    "- trace.csv: 0 rows in window; no timestamps or CMDB ids present in this window.\n"
    "- log.csv: 0 rows in window; no timestamps or CMDB ids present in this window.\n"
    "- error_logs.csv: 0 rows in window; no timestamps or CMDB ids present in this window.\n\n"
    "Additional note: metric aggregation for the window is empty (no (cmdb_id, kpi_name) groups found).\n"
    "If you want, I can search a different time window or show the nearest available timestamps around this window."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(        file_name  rows_in_window min_timestamp_in_window max_timestamp_in_window sorted_unique_cmdb_ids_in_window
0      metric.csv               0                    None                    None                               []
1       trace.csv               0                    None                    None                               []
2         log.csv               0                    None                    None                               []
3  error_logs.csv               0                    None                    None                               [], Empty DataFrame
Columns: [cmdb_id, kpi_name, count]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing dataframes and tz if present; otherwise load
try:
    metric_df
    trace_df
    log_df
    error_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

# window bounds in UTC+8
window_start = pd.Timestamp('2021-03-07 01:30:00', tz=tz)
window_end = pd.Timestamp('2021-03-07 02:00:00', tz=tz)

def localized_ts_series(df):
    # return tz-aware series; handle missing timestamp column or empty df
    if 'timestamp' not in df.columns or df.empty:
        return pd.Series([], dtype='datetime64[ns]')
    return pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

def file_context_summary(df, file_name, sample_limit=20):
    ts = localized_ts_series(df)
    # global min/max
    if ts.empty:
        global_min = None
        global_max = None
    else:
        global_min = ts.min().isoformat()
        global_max = ts.max().isoformat()
    # nearest before window: max ts < window_start
    if ts.empty or (ts < window_start).sum() == 0:
        before_ts_iso = None
        before_count = 0
        before_cmdbs = []
    else:
        before_ts = ts[ts < window_start].max()
        before_ts_iso = before_ts.isoformat()
        # count rows equal to that timestamp
        before_mask = ts == before_ts
        before_count = int(before_mask.sum())
        before_cmdbs = sorted(df.loc[before_mask, 'cmdb_id'].dropna().unique())[:sample_limit]
    # nearest after window: min ts > window_end
    if ts.empty or (ts > window_end).sum() == 0:
        after_ts_iso = None
        after_count = 0
        after_cmdbs = []
    else:
        after_ts = ts[ts > window_end].min()
        after_ts_iso = after_ts.isoformat()
        after_mask = ts == after_ts
        after_count = int(after_mask.sum())
        after_cmdbs = sorted(df.loc[after_mask, 'cmdb_id'].dropna().unique())[:sample_limit]
    return {
        'file_name': file_name,
        'global_min_timestamp': global_min,
        'global_max_timestamp': global_max,
        'nearest_before_window_timestamp': before_ts_iso,
        'count_rows_at_nearest_before': before_count,
        'sample_cmdb_ids_at_nearest_before': before_cmdbs,
        'nearest_after_window_timestamp': after_ts_iso,
        'count_rows_at_nearest_after': after_count,
        'sample_cmdb_ids_at_nearest_after': after_cmdbs
    }

rows = [
    file_context_summary(metric_df, 'metric.csv'),
    file_context_summary(trace_df, 'trace.csv'),
    file_context_summary(log_df, 'log.csv'),
    file_context_summary(error_df, 'error_logs.csv')
]

window_context_df = pd.DataFrame(rows)

# Display compact single-row-per-file summary
window_context_df

```
Out[3]:
```
Summary (all times UTC+8):

- metric.csv
  - Global time range: 2021-03-06T17:30:00 → 2021-03-06T18:00:00
  - Nearest timestamp before the incident window: 2021-03-06T18:00:00
  - Rows at that timestamp: 973
  - Sample CMDB ids at that timestamp (up to 20): IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, Tomcat01–Tomcat04, ... 
  - No timestamp was found after the incident window.

- trace.csv
  - Global time range: 2021-03-06T17:30:00 → 2021-03-06T17:59:00
  - Nearest timestamp before the incident window: 2021-03-06T17:59:00
  - Rows at that timestamp: 204
  - Sample CMDB ids at that timestamp (up to 20): IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, Tomcat03, Tomcat04, ...
  - No timestamp was found after the incident window.

- log.csv
  - Global time range: 2021-03-06T17:30:00 → 2021-03-06T18:00:00
  - Nearest timestamp before the incident window: 2021-03-06T18:00:00
  - Rows at that timestamp: 8
  - Sample CMDB ids at that timestamp (up to 20): Mysql01, Tomcat03, Tomcat04, apache02
  - No timestamp was found after the incident window.

- error_logs.csv
  - No timestamps or rows present in this file (empty).
  - Therefore no before/after timestamps or CMDB ids.

If you want, I can list the full set of CMDB ids seen at each nearest timestamp or show the exact rows for those timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

file_name       global_min_timestamp       global_max_timestamp nearest_before_window_timestamp  count_rows_at_nearest_before                  sample_cmdb_ids_at_nearest_before nearest_after_window_timestamp  count_rows_at_nearest_after sample_cmdb_ids_at_nearest_after
0      metric.csv  2021-03-06T17:30:00+08:00  2021-03-06T18:00:00+08:00       2021-03-06T18:00:00+08:00                           973  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...                           None                            0                               []
1       trace.csv  2021-03-06T17:30:00+08:00  2021-03-06T17:59:00+08:00       2021-03-06T17:59:00+08:00                           204  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...                           None                            0                               []
2         log.csv  2021-03-06T17:30:00+08:00  2021-03-06T18:00:00+08:00       2021-03-06T18:00:00+08:00                             8            [Mysql01, Tomcat03, Tomcat04, apache02]                           None                            0                               []
3  error_logs.csv                       None                       None                            None                             0                                                 []                           None                            0                               []```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse dataframes and tz from kernel state
tz = pytz.timezone('Asia/Shanghai')

# Ensure dataframes exist
# metric_df, trace_df, log_df, error_df are expected to be in the namespace from previous runs

# Helper to create tz-localized timestamp series consistent with prior steps
def tz_localized_series(df):
    if 'timestamp' not in df.columns or df.empty:
        return pd.Series(pd.to_datetime([], unit='s'))
    return pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

# Define target nearest-before timestamps (as tz-aware)
metric_target_ts = pd.Timestamp('2021-03-06 18:00:00', tz=tz)
trace_target_ts  = pd.Timestamp('2021-03-06 17:59:00', tz=tz)
log_target_ts    = pd.Timestamp('2021-03-06 18:00:00', tz=tz)

# 1) metric.csv anomaly scan
# prepare series and numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_ts = tz_localized_series(metric_df)

# compute global p95 and p5 per (cmdb_id, kpi_name) using full series (before filtering)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# extract rows at the nearest-before timestamp
metric_mask = metric_ts == metric_target_ts
metric_at_ts = metric_df[metric_mask].copy()
metric_at_ts = metric_at_ts[['cmdb_id', 'kpi_name', 'value']]

if not metric_at_ts.empty:
    # merge thresholds
    metric_merged = metric_at_ts.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    # decide anomaly: value >= p95 OR value <= p5
    cond_high = metric_merged['value'] >= metric_merged['global_p95']
    cond_low  = metric_merged['value'] <= metric_merged['global_p5']
    metric_merged = metric_merged[cond_high | cond_low].copy()
    if not metric_merged.empty:
        # compute severity with safe division
        def compute_sev(row):
            v = row['value']
            p95 = row['global_p95']
            p5 = row['global_p5']
            # handle NaN or zero thresholds
            if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
                return np.nan
            if v >= p95:
                if p95 == 0:
                    return np.inf
                return v / p95
            else:
                # v <= p5
                if v == 0:
                    if p5 == 0:
                        return np.inf
                    else:
                        return p5 / 1e-12  # large
                return p5 / v
        metric_merged['severity'] = metric_merged.apply(compute_sev, axis=1)
        metric_anomalies = (
            metric_merged
            .rename(columns={'value':'value_at_timestamp'})
            .loc[:, ['cmdb_id','kpi_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series']]
            .sort_values('severity', ascending=False)
            .head(20)
            .reset_index(drop=True)
        )
    else:
        metric_anomalies = pd.DataFrame(columns=['cmdb_id','kpi_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series'])
else:
    metric_anomalies = pd.DataFrame(columns=['cmdb_id','kpi_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series'])

# 2) trace.csv anomaly scan
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_ts = tz_localized_series(trace_df)

trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

trace_mask = trace_ts == trace_target_ts
trace_at_ts = trace_df[trace_mask].copy()
trace_at_ts = trace_at_ts[['cmdb_id','trace_name','value']]

if not trace_at_ts.empty:
    trace_merged = trace_at_ts.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    cond_high = trace_merged['value'] >= trace_merged['global_p95']
    cond_low  = trace_merged['value'] <= trace_merged['global_p5']
    trace_merged = trace_merged[cond_high | cond_low].copy()
    if not trace_merged.empty:
        def compute_sev_row(row):
            v = row['value']; p95 = row['global_p95']; p5 = row['global_p5']
            if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
                return np.nan
            if v >= p95:
                return np.inf if p95 == 0 else v / p95
            else:
                if v == 0:
                    return np.inf if p5 == 0 else p5 / 1e-12
                return p5 / v
        trace_merged['severity'] = trace_merged.apply(compute_sev_row, axis=1)
        trace_anomalies = (
            trace_merged
            .rename(columns={'value':'value_at_timestamp'})
            .loc[:, ['cmdb_id','trace_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series']]
            .sort_values('severity', ascending=False)
            .head(20)
            .reset_index(drop=True)
        )
    else:
        trace_anomalies = pd.DataFrame(columns=['cmdb_id','trace_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series'])
else:
    trace_anomalies = pd.DataFrame(columns=['cmdb_id','trace_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series'])

# 3) log.csv anomaly scan
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_ts = tz_localized_series(log_df)

log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

log_mask = log_ts == log_target_ts
log_at_ts = log_df[log_mask].copy()
log_at_ts = log_at_ts[['cmdb_id','log_name','value']]

if not log_at_ts.empty:
    log_merged = log_at_ts.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    cond_high = log_merged['value'] >= log_merged['global_p95']
    cond_low  = log_merged['value'] <= log_merged['global_p5']
    log_merged = log_merged[cond_high | cond_low].copy()
    if not log_merged.empty:
        def compute_sev_log(row):
            v = row['value']; p95 = row['global_p95']; p5 = row['global_p5']
            if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
                return np.nan
            if v >= p95:
                return np.inf if p95 == 0 else v / p95
            else:
                if v == 0:
                    return np.inf if p5 == 0 else p5 / 1e-12
                return p5 / v
        log_merged['severity'] = log_merged.apply(compute_sev_log, axis=1)
        log_anomalies = (
            log_merged
            .rename(columns={'value':'value_at_timestamp'})
            .loc[:, ['cmdb_id','log_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series']]
            .sort_values('severity', ascending=False)
            .head(20)
            .reset_index(drop=True)
        )
    else:
        log_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series'])
else:
    log_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series'])

# 4) error_logs.csv confirmation
# error_df is expected empty from previous steps; create an empty placeholder result
error_anomalies = pd.DataFrame(columns=['note'])
if error_df.empty or 'timestamp' not in error_df.columns:
    error_anomalies = pd.DataFrame([{'note': 'error_logs.csv is empty; no anomalies.'}])
else:
    # If not empty (unexpected), find same nearest-before timestamp behavior
    error_anomalies = pd.DataFrame([{'note': 'error_logs.csv contains data; fallback scan not implemented for textual messages in this task.'}])

# If no anomalies found in a category, make a small DataFrame stating that explicitly
if metric_anomalies.empty:
    metric_anomalies = pd.DataFrame([{
        'cmdb_id': None,
        'kpi_name': None,
        'value_at_timestamp': None,
        'global_p95': None,
        'global_p5': None,
        'severity': None,
        'count_full_series': 0,
        'note': 'No anomalies found at nearest-before timestamp (fallback inference).'
    }]).loc[:, ['cmdb_id','kpi_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series','note']]

if trace_anomalies.empty:
    trace_anomalies = pd.DataFrame([{
        'cmdb_id': None,
        'trace_name': None,
        'value_at_timestamp': None,
        'global_p95': None,
        'global_p5': None,
        'severity': None,
        'count_full_series': 0,
        'note': 'No anomalies found at nearest-before timestamp (fallback inference).'
    }]).loc[:, ['cmdb_id','trace_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series','note']]

if log_anomalies.empty:
    log_anomalies = pd.DataFrame([{
        'cmdb_id': None,
        'log_name': None,
        'value_at_timestamp': None,
        'global_p95': None,
        'global_p5': None,
        'severity': None,
        'count_full_series': 0,
        'note': 'No anomalies found at nearest-before timestamp (fallback inference).'
    }]).loc[:, ['cmdb_id','log_name','value_at_timestamp','global_p95','global_p5','severity','count_full_series','note']]

# Display compact outputs: top anomalies per file (or explicit no-anomaly row) and error note
metric_anomalies, trace_anomalies, log_anomalies, error_anomalies

```
Out[4]:
```
Fallback inference (scanning the nearest-before timestamps because the incident window had no telemetry)

Summary of findings and interpretation:

1) metric.csv (timestamp scanned: 2021-03-06 18:00:00 UTC+8)
- Many (cmdb_id, kpi_name) groups flagged because the value at the scanned timestamp was extreme vs the group’s historical distribution (global P95 / P5).
- Top candidates are dominated by Mysql02 metrics and some OSLinux/local-disk metrics. Examples:
  - IG01: OSLinux local-disk DSKPercent-like KPI — value = 0.0, global_p95 ≈ 0.2, global_p5 = 0.0 → severity reported as infinite (division by zero in threshold). count_full_series for groups ~15–31.
  - Many Mysql02 MySQL and OSLinux process/disk KPIs — value = 0.0 while historical P95 or P5 are non-zero or thresholds are zero → severity reported as infinite.
- Interpretation: either an extreme drop-to-zero was observed for these metrics at that timestamp (possible real drop or a metric collection failure), or the P5/P95 thresholds include zeros causing very large/infinite severity. Most flagged groups have robust historical counts (many with 20–31 points).

2) trace.csv (timestamp scanned: 2021-03-06 17:59:00 UTC+8)
- Several latency/duration trace features are elevated at the scanned timestamp compared with historical P95/P5.
- Top examples:
  - MG01 trace.from_Tomcat02.duration_mean and Tomcat02 trace.to_MG01.duration_mean — value ≈ 0.2845 vs global_p95 ≈ 0.1229 (severity ≈ 2.31).
  - Tomcat02 ↔ IG02 related duration_p95/mean entries also show elevated values (severity ~1.6).
- Interpretation: these anomalies point to increased RPC/trace durations involving Tomcat02, MG01 and IG02 at the scanned time. Counts used to compute thresholds are ~29–30, so thresholds are based on adequate history.

3) log.csv (timestamp scanned: 2021-03-06 18:00:00 UTC+8)
- Two anomaly patterns:
  - log.error_count entries: value = 0 and global thresholds are 0 → reported severity infinite (artifact of zero thresholds). These appear for Mysql01, Tomcat03, apache02, Tomcat04 (counts vary).
  - log.row_count entries: notably low row_count values relative to historical P5/P95:
    - apache02 log.row_count: value = 2.0, global_p95 ≈ 61.5, global_p5 ≈ 10.0 → severity = 10/2 = 5.0
    - Tomcat04 log.row_count: value = 1.0, global_p95 ≈ 31.0, global_p5 ≈ 4.5 → severity = 4.5/1 = 4.5
    - Tomcat03 log.row_count: value = 1.0, global_p95 ≈ 37.0, global_p5 ≈ 4.0 → severity = 4.0/1 = 4.0
  - Interpretation: these indicate a sharp drop in log row_count (likely traffic or logging volume) for several front-end/service nodes at the scanned time. The error_count zeros may be normal (no errors) or threshold artifacts.

4) error_logs.csv
- Confirmed empty; no anomalies.

Overall takeaway (fallback inference):
- The fallback scan (nearest-before timestamps) surfaces two main signals:
  1) Trace latency increases involving Tomcat02 ↔ MG01/IG02 (likely real and supported by good historical counts).
  2) Several metric/log groups show zero or greatly reduced values at that timestamp (many for Mysql02 and some disk/process metrics, and reduced log row_count for apache/Tomcat). Some of these produce infinite severity due to zero thresholds — treat those cautiously (could be monitoring gaps or true zero activity).
- Recommendation: investigate Tomcat02 ↔ MG01/IG02 latencies first (trace-level slowdowns). Also validate collection health for the metrics that drop to zero (check agent/collection and MySQL/service availability) and confirm whether log row_count drops correspond to traffic loss or logging pipeline issues.

If you want, I can:
- Show the top N anomaly rows in full for any category, or
- Retrieve raw rows around the scanned timestamps for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  value_at_timestamp  global_p95  global_p5  severity  count_full_series
0      IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                 0.0         0.2        0.0       inf                 16
1   Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                 0.0         0.0        0.0       inf                 31
2   Mysql02                   Mysql-MySQL_3306_Threads Created                 0.0         0.0        0.0       inf                 20
3   Mysql02         Mysql-MySQL_3306_max trx lock memory bytes                 0.0        56.8        0.0       inf                 20
4   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                 0.0         1.2        0.0       inf                 15
5   Mysql02                 Mysql-MySQL_3306_Innodb data reads                 0.0         0.0        0.0       inf                 20
6   Mysql02                  Mysql-MySQL_3306_Innodb log waits                 0.0         0.0        0.0       inf                 20
7   Mysql02                Mysql-MySQL_3306_Select Range Check                 0.0         0.0        0.0       inf                 20
8   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                 0.0         0.0        0.0       inf                 15
9   Mysql02                        Mysql-MySQL_3306_Sort Range                 0.0         0.0        0.0       inf                 20
10  Mysql02                         Mysql-MySQL_3306_Sort Scan                 0.0         0.0        0.0       inf                 20
11  Mysql02                Mysql-MySQL_3306_Table Locks Waited                 0.0         0.0        0.0       inf                 20
12  Mysql02        Mysql-MySQL_3306_Table open cache overflows                 0.0         0.0        0.0       inf                 20
13  Mysql02          Mysql-MySQL_3306_Opened table definitions                 0.0         0.0        0.0       inf                 20
14  Mysql02                       Mysql-MySQL_3306_Qcache Hits                 0.0         0.0        0.0       inf                 20
15  Mysql02             Mysql-MySQL_3306_Tc log max pages used                 0.0         0.0        0.0       inf                 20
16  Mysql02                 Mysql-MySQL_3306_Tc log page waits                 0.0         0.0        0.0       inf                 20
17  Mysql02                  Mysql-MySQL_3306_Select Full Join                 0.0         0.0        0.0       inf                 20
18  Mysql02            Mysql-MySQL_3306_Select Full Range Join                 0.0         0.0        0.0       inf                 20
19  Mysql02                      Mysql-MySQL_3306_Select Range                 0.0         0.0        0.0       inf                 20,      cmdb_id                         trace_name  value_at_timestamp  global_p95  global_p5  severity  count_full_series
0       MG01  trace.from_Tomcat02.duration_mean            0.284500    0.122900   0.014337  2.314893                 30
1   Tomcat02        trace.to_MG01.duration_mean            0.284500    0.122900   0.014337  2.314893                 30
2   Tomcat02       trace.from_IG02.duration_p95            0.458050    0.282310   0.026230  1.622507                 30
3       IG02     trace.to_Tomcat02.duration_p95            0.458050    0.282310   0.026230  1.622507                 30
4   Tomcat02      trace.from_IG02.duration_mean            0.149000    0.095127   0.019477  1.566330                 30
5       IG02    trace.to_Tomcat02.duration_mean            0.149000    0.095127   0.019477  1.566330                 30
6   Tomcat02         trace.to_MG01.duration_p95            0.502750    0.378170   0.025263  1.329429                 30
7       MG01   trace.from_Tomcat02.duration_p95            0.502750    0.378170   0.025263  1.329429                 30
8   Tomcat02        trace.to_MG02.duration_mean            0.011143    0.102565   0.011304  1.014417                 30
9       MG02  trace.from_Tomcat02.duration_mean            0.011143    0.102565   0.011304  1.014417                 30
10      IG02    trace.to_Tomcat03.duration_mean            0.023857    0.136712   0.024071  1.008978                 30
11  Tomcat03      trace.from_IG02.duration_mean            0.023857    0.136712   0.024071  1.008978                 30
12      MG02        trace.to_dockerA2.row_count            1.000000   14.550000   1.000000  1.000000                 30
13      MG01  trace.from_dockerB2.duration_mean            0.006000    0.060157   0.006000  1.000000                 29
14      MG01      trace.from_Tomcat02.row_count            2.000000   17.550000   2.000000  1.000000                 30
15  Tomcat02            trace.to_MG01.row_count            2.000000   17.550000   2.000000  1.000000                 30
16  dockerA1            trace.self.duration_p95            0.001000    0.009000   0.001000  1.000000                 30
17  dockerA2          trace.from_MG02.row_count            1.000000   14.550000   1.000000  1.000000                 30
18  dockerA2            trace.self.duration_p95            0.002000    0.010117   0.002000  1.000000                 30
19  dockerB2        trace.to_MG01.duration_mean            0.006000    0.060157   0.006000  1.000000                 29,     cmdb_id         log_name  value_at_timestamp  global_p95  global_p5  severity  count_full_series
0   Mysql01  log.error_count                 0.0         0.0        0.0       inf                  3
1  Tomcat03  log.error_count                 0.0         0.0        0.0       inf                 31
2  apache02  log.error_count                 0.0         0.0        0.0       inf                 31
3  Tomcat04  log.error_count                 0.0         0.0        0.0       inf                 31
4  apache02    log.row_count                 2.0        61.5       10.0  5.000000                 31
5  Tomcat04    log.row_count                 1.0        31.0        4.5  4.500000                 31
6  Tomcat03    log.row_count                 1.0        37.0        4.0  4.000000                 31
7   Mysql01    log.row_count                 4.0         3.8        1.1  1.052632                  3,                                      note
0  error_logs.csv is empty; no anomalies.)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# reuse existing dataframes and tz if available
try:
    metric_df
    trace_df
    log_df
    error_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# define window bounds (15 minutes before nearest-before timestamp)
window_start = pd.Timestamp('2021-03-06 17:45:00', tz=tz)
window_end   = pd.Timestamp('2021-03-06 18:00:00', tz=tz)

# helper to get tz-localized timestamp series
def localized_ts(df):
    if 'timestamp' not in df.columns or df.empty:
        return pd.Series(pd.to_datetime([], unit='s'))
    return pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

# 1) Metric: filter for selected cmdb_ids and time window
metric_targets = ["Tomcat02","MG01","IG02","Mysql02","apache02","Tomcat03","Tomcat04"]
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_ts = localized_ts(metric_df)
metric_mask = metric_df['cmdb_id'].isin(metric_targets) & (metric_ts >= window_start) & (metric_ts <= window_end)
metric_window = metric_df[metric_mask].copy()

if not metric_window.empty:
    metric_window['timestamp_iso'] = pd.to_datetime(metric_window['timestamp'], unit='s').dt.tz_localize(tz).astype(str)
    metric_window_rows = metric_window.loc[:, ['timestamp_iso','timestamp','cmdb_id','kpi_name','value']].sort_values('timestamp', ascending=False).head(200)
    # Aggregation by (cmdb_id,kpi_name)
    metric_window_agg = (
        metric_window
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(count_in_window='count',
             min_in_window='min',
             max_in_window='max',
             mean_in_window='mean')
        .sort_values('count_in_window', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )
else:
    metric_window_rows = pd.DataFrame(columns=['timestamp_iso','timestamp','cmdb_id','kpi_name','value'])
    metric_window_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','min_in_window','max_in_window','mean_in_window'])

# 2) Trace: filter for selected cmdb_ids and time window
trace_targets = ["Tomcat02","MG01","IG02","Tomcat03","MG02","dockerA2","dockerB2"]
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_ts = localized_ts(trace_df)
trace_mask = trace_df['cmdb_id'].isin(trace_targets) & (trace_ts >= window_start) & (trace_ts <= window_end)
trace_window = trace_df[trace_mask].copy()

if not trace_window.empty:
    trace_window['timestamp_iso'] = pd.to_datetime(trace_window['timestamp'], unit='s').dt.tz_localize(tz).astype(str)
    trace_window_rows = trace_window.loc[:, ['timestamp_iso','timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=False).head(200)
else:
    trace_window_rows = pd.DataFrame(columns=['timestamp_iso','timestamp','cmdb_id','trace_name','value'])

# 3) Log: filter for selected cmdb_ids and time window
log_targets = ["apache02","Tomcat03","Tomcat04","Mysql01"]
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_ts = localized_ts(log_df)
log_mask = log_df['cmdb_id'].isin(log_targets) & (log_ts >= window_start) & (log_ts <= window_end)
log_window = log_df[log_mask].copy()

if not log_window.empty:
    log_window['timestamp_iso'] = pd.to_datetime(log_window['timestamp'], unit='s').dt.tz_localize(tz).astype(str)
    log_window_rows = log_window.loc[:, ['timestamp_iso','timestamp','cmdb_id','log_name','value']].sort_values('timestamp', ascending=False).head(200)
else:
    log_window_rows = pd.DataFrame(columns=['timestamp_iso','timestamp','cmdb_id','log_name','value'])

# Label
inspection_label = "15-minute-before-nearest-before-window inspection (2021-03-06 17:45:00 to 2021-03-06 18:00:00 UTC+8)"

# Display compact results: label and the three tables (limited in size)
inspection_label, metric_window_rows, metric_window_agg, trace_window_rows, log_window_rows

```
Out[5]:
```
summary_text = (
    "15-minute-before-nearest-before-window inspection (2021-03-06 17:45 → 18:00 UTC+8) — concise findings:\n\n"
    "Metric (filtered components: Tomcat02, MG01, IG02, Mysql02, apache02, Tomcat03, Tomcat04):\n"
    "- Data present: up to 200 recent rows returned (sample shows many points at 18:00). Many metrics have value==0 at 18:00 (notably Mysql02 and some disk/process KPIs).\n"
    "- Aggregates: most (cmdb_id,kpi_name) groups in this window have count=16 (full 15-min coverage at 1-min granularity). Several session/request/thread KPIs for Tomcat02 have constant values (e.g., CurrentThreads=100).\n"
    "- Interpretation: presence of many zeros at the 18:00 timestamp (Mysql02, local-disk, some Tomcat session KPIs) is consistent with the earlier anomaly hits — could be a genuine drop or collection gap. Verify metric collection/agent and service availability for Mysql02 and disk metrics.\n\n"
    "Trace (filtered components: Tomcat02, MG01, IG02, Tomcat03, MG02, dockerA2, dockerB2):\n"
    "- Data present: trace rows exist across the 15-min window (sample shows entries at 17:59 and 17:58).\n"
    "- Observed: elevated durations involving Tomcat02 ↔ MG01/IG02 around 17:59 (matches fallback anomaly scan). Counts indicate ~29–30 history points used for thresholds.\n"
    "- Interpretation: real trace-level latency increases around Tomcat02 → investigate Tomcat02, MG01, IG02 service interactions and network between them.\n\n"
    "Log (filtered components: apache02, Tomcat03, Tomcat04, Mysql01):\n"
    "- Data present: log rows returned (103 rows in the window). At 18:00, examples: Mysql01 log.row_count=4.0, Tomcat03 log.row_count=1.0/log.error_count=0, Tomcat04 log.row_count=1.0.\n"
    "- Observed: marked drop in log.row_count for apache02/Tomcat nodes at the end of the window (and small or zero error_count values). This aligns with metric drops for some components.\n"
    "- Interpretation: log volume drop suggests reduced traffic or logging ingestion issues; cross-check upstream traffic and logging pipeline.\n\n"
    "Overall actionable summary:\n"
    "1) Prioritize investigating trace latency: Tomcat02 ↔ MG01/IG02 (trace anomalies around 17:59).\n"
    "2) Validate metric collection and service health for Mysql02 and local-disk/process metrics (many zero values at 18:00 can indicate collection failure or real service impact).\n"
    "3) Correlate the reduced log.row_count for apache02/Tomcat03/Tomcat04 with traffic or logging pipeline problems.\n\n"
    "If you want, I can: (A) show the top N aggregated metric groups from the window, (B) display the full filtered rows for a specific component and timestamp, or (C) fetch raw trace rows around 17:58–18:00 for Tomcat02/MG01/IG02."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

('15-minute-before-nearest-before-window inspection (2021-03-06 17:45:00 to 2021-03-06 18:00:00 UTC+8)',                    timestamp_iso   timestamp   cmdb_id                                           kpi_name         value
27549  2021-03-06 18:00:00+08:00  1615053600  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  0.000000e+00
27544  2021-03-06 18:00:00+08:00  1615053600  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00
27580  2021-03-06 18:00:00+08:00  1615053600  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02
27664  2021-03-06 18:00:00+08:00  1615053600  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.252277e+06
27565  2021-03-06 18:00:00+08:00  1615053600  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.614060e+05
...                          ...         ...       ...                                                ...           ...
27157  2021-03-06 18:00:00+08:00  1615053600   Mysql02              Mysql-MySQL_3306_Innodb os log fsyncs  8.083300e+00
27110  2021-03-06 18:00:00+08:00  1615053600   Mysql02           Mysql-MySQL_3306_Created Tmp Disk Tables  0.000000e+00
27111  2021-03-06 18:00:00+08:00  1615053600   Mysql02                 Mysql-MySQL_3306_Created Tmp Files  0.000000e+00
27112  2021-03-06 18:00:00+08:00  1615053600   Mysql02                Mysql-MySQL_3306_Created Tmp Tables  5.000000e-02
27129  2021-03-06 18:00:00+08:00  1615053600   Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time  0.000000e+00

[200 rows x 5 columns],      cmdb_id                                           kpi_name  count_in_window  min_in_window  max_in_window  mean_in_window
0   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...               16         0.0000       0.000000         0.00000
1   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...               16         0.0000       0.000000         0.00000
2   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...               16         0.0000       0.000000         0.00000
3   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...               16       389.0000     389.000000       389.00000
4   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...               16         0.0000       0.000000         0.00000
..       ...                                                ...              ...            ...            ...             ...
45   Mysql02                         OSLinux-CPU_CPU_CPUSysTime               16         0.3488       0.581395         0.45082
46   Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               16         0.5000       0.500000         0.50000
47  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...               16         0.0000       0.000000         0.00000
48  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               16       100.0000     100.000000       100.00000
49  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               16         0.0000       0.000000         0.00000

[50 rows x 6 columns],                   timestamp_iso   timestamp   cmdb_id                         trace_name     value
6388  2021-03-06 17:59:00+08:00  1615053540  Tomcat02            trace.self.duration_p95  0.015100
6349  2021-03-06 17:59:00+08:00  1615053540      MG02   trace.from_dockerB2.duration_p95  0.007400
6400  2021-03-06 17:59:00+08:00  1615053540  Tomcat03       trace.from_IG02.duration_p95  0.044600
6473  2021-03-06 17:59:00+08:00  1615053540  dockerB2            trace.to_MG02.row_count  7.000000
6472  2021-03-06 17:59:00+08:00  1615053540  dockerB2         trace.to_MG02.duration_p95  0.007400
...                         ...         ...       ...                                ...       ...
6116  2021-03-06 17:58:00+08:00  1615053480      MG01      trace.from_Tomcat01.row_count  7.000000
6117  2021-03-06 17:58:00+08:00  1615053480      MG01  trace.from_Tomcat02.duration_mean  0.019167
6102  2021-03-06 17:58:00+08:00  1615053480      IG02    trace.to_Tomcat01.duration_mean  0.016500
6103  2021-03-06 17:58:00+08:00  1615053480      IG02     trace.to_Tomcat01.duration_p95  0.016950
6215  2021-03-06 17:58:00+08:00  1615053480  Tomcat03            trace.to_MG02.row_count  3.000000

[200 rows x 5 columns],                  timestamp_iso   timestamp   cmdb_id         log_name  value
377  2021-03-06 18:00:00+08:00  1615053600   Mysql01    log.row_count    4.0
378  2021-03-06 18:00:00+08:00  1615053600  Tomcat03  log.error_count    0.0
379  2021-03-06 18:00:00+08:00  1615053600  Tomcat03    log.row_count    1.0
380  2021-03-06 18:00:00+08:00  1615053600  Tomcat04  log.error_count    0.0
381  2021-03-06 18:00:00+08:00  1615053600  Tomcat04    log.row_count    1.0
..                         ...         ...       ...              ...    ...
195  2021-03-06 17:45:00+08:00  1615052700  Tomcat04    log.row_count    7.0
198  2021-03-06 17:45:00+08:00  1615052700  apache02  log.error_count    0.0
199  2021-03-06 17:45:00+08:00  1615052700  apache02    log.row_count   18.0
193  2021-03-06 17:45:00+08:00  1615052700  Tomcat03    log.row_count   12.0
192  2021-03-06 17:45:00+08:00  1615052700  Tomcat03  log.error_count    0.0

[103 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df and tz if available; otherwise load
try:
    trace_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    trace_df = pd.read_csv('trace.csv')

# ensure value numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using the entire series
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# 2) Filter to the 15-min window and selected cmdb_ids
tz = pytz.timezone('Asia/Shanghai')
window_start = pd.Timestamp('2021-03-06 17:45:00', tz=tz)
window_end   = pd.Timestamp('2021-03-06 18:00:00', tz=tz)
selected_cmdbs = ["Tomcat02","MG01","IG02"]

# create tz-aware timestamp column for comparisons
trace_df['ts_local'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)

mask_window = (trace_df['ts_local'] >= window_start) & (trace_df['ts_local'] <= window_end) & (trace_df['cmdb_id'].isin(selected_cmdbs))
trace_window = trace_df[mask_window].copy()

# 3) For each group present in the filtered window, find rows where value >= global_p95 OR value <= global_p5
if trace_window.empty:
    label = "earliest trace anomalies (Tomcat02/MG01/IG02) in 15-min window"
    result_df = pd.DataFrame([{
        'note': 'No trace rows for selected components in the 15-min window; no anomalies found.'
    }])
else:
    # merge global thresholds into the windowed rows
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    # drop rows with NaN values (cannot evaluate)
    tw = tw.dropna(subset=['value'])
    # condition flags
    cond_high = tw['value'] >= tw['global_p95']
    cond_low  = tw['value'] <= tw['global_p5']
    tw_matched = tw[cond_high | cond_low].copy()
    if tw_matched.empty:
        label = "earliest trace anomalies (Tomcat02/MG01/IG02) in 15-min window"
        result_df = pd.DataFrame([{
            'note': 'No trace anomalies (value >= global_p95 or <= global_p5) found for selected components in the 15-min window.'
        }])
    else:
        # compute severity per row with safe handling of zeros/NaN
        def compute_severity(row):
            v = row['value']
            p95 = row['global_p95']
            p5 = row['global_p5']
            if pd.isna(v):
                return np.nan
            if not pd.isna(p95) and v >= p95:
                if p95 == 0:
                    return np.inf
                return v / p95
            else:
                # v <= p5 branch
                if pd.isna(p5):
                    return np.nan
                if v == 0:
                    # if both zero, treat as inf severity (artifact); else very large
                    return np.inf if p5 == 0 else p5 / 1e-12
                return p5 / v

        tw_matched['severity'] = tw_matched.apply(compute_severity, axis=1)

        # For each (cmdb_id, trace_name) pick earliest timestamp in-window where condition met
        idx_earliest = tw_matched.groupby(['cmdb_id','trace_name'])['ts_local'].idxmin()
        earliest_rows = tw_matched.loc[idx_earliest].copy()

        # build result columns
        result_df = earliest_rows.loc[:, [
            'cmdb_id',
            'trace_name',
            'ts_local',
            'value',
            'global_p95',
            'global_p5',
            'severity',
            'count_full_series'
        ]].rename(columns={
            'ts_local': 'earliest_timestamp_in_window',
            'value': 'value_at_timestamp'
        })

        # sort by earliest timestamp ascending and limit to top 20
        result_df = result_df.sort_values('earliest_timestamp_in_window', ascending=True).head(20).reset_index(drop=True)

        # convert timestamps to ISO strings for compact display
        result_df['earliest_timestamp_in_window'] = result_df['earliest_timestamp_in_window'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')

        label = "earliest trace anomalies (Tomcat02/MG01/IG02) in 15-min window"

# Display label and compact result (limited to 20 rows)
label, result_df

```
Out[6]:
```
Label: earliest trace anomalies (Tomcat02/MG01/IG02) in 15-min window

Summary (plain English):

- Overview: Anomaly candidates were found between 17:45 and 17:47 (UTC+8). Earliest anomalies start at 2021-03-06T17:45:00+08:00 for MG01, Tomcat02 and IG02-related traces.

- Earliest/high-priority items at 17:45:
  - MG01 / trace.from_dockerB1.duration_p95 — value 0.03065 vs global_p95 0.02649 (severity ≈ 1.157). (count_full_series = 29)
  - MG01 / trace.from_Tomcat04.row_count — value 3 vs global_p95 15, global_p5 3 (severity ≈ 1.13). (count_full_series = 29)
  - Tomcat02 / trace.from_IG01.row_count — value 3 vs global_p5 3 (severity = 1.0). (count_full_series = 30)
  - IG02 / trace.to_Tomcat04.row_count — value 4 vs global_p5 4 (severity = 1.0). (count_full_series = 30)
  - Tomcat02 / trace.from_IG02.row_count and IG02 / trace.to_Tomcat02.row_count — both value 3 at 17:45 (severity = 1.0). (count_full_series = 30)
  - Tomcat02 / trace.from_IG02.duration_mean and trace.self.duration_mean/p95 show modest elevation (severity ~1.02–1.07).

- Notable later/stronger deviations (17:46):
  - MG01 / trace.to_dockerA1.duration_p95 — value 0.0217 vs global_p95 0.21887 and global_p5 0.03210 (severity ≈ 1.48). (count_full_series = 29)
  - Several other MG01 and IG02 row_count/duration entries at 17:46–17:47 showing deviations (many have severity ≈ 1.0–1.21).

- Interpretation:
  - The anomalies are mostly low-to-moderate severity (severity ~1.0–1.48), with many being row_count drops (values at or near global P5) and some modest duration increases.
  - Earliest signals concentrate at 17:45 — focus initial investigation on MG01 and Tomcat02 interactions around 17:45 (row_count drops and modest latency increases).

- Suggested next steps:
  - Inspect traces and logs for MG01 and Tomcat02 around 17:45 to confirm whether row_count drops reflect reduced traffic or sampling/collection issues.
  - Correlate with metrics and log volume (you already saw metric/log drops at 18:00); check network/queueing between Tomcat02 ↔ MG01/IG02.

The original code execution output of IPython Kernel is also provided below for reference:

('earliest trace anomalies (Tomcat02/MG01/IG02) in 15-min window',      cmdb_id                         trace_name earliest_timestamp_in_window  value_at_timestamp  global_p95  global_p5  severity  count_full_series
0       MG01   trace.from_dockerB1.duration_p95     2021-03-06T17:45:00+0800            0.030650    0.026490   0.006280  1.157040                 29
1       MG01      trace.from_Tomcat04.row_count     2021-03-06T17:45:00+0800            3.000000   15.000000   3.400000  1.133333                 29
2   Tomcat02          trace.from_IG01.row_count     2021-03-06T17:45:00+0800            3.000000   17.000000   3.000000  1.000000                 30
3       IG02        trace.to_Tomcat04.row_count     2021-03-06T17:45:00+0800            4.000000   15.650000   4.000000  1.000000                 30
4   Tomcat02          trace.from_IG02.row_count     2021-03-06T17:45:00+0800            3.000000   17.100000   3.000000  1.000000                 30
5       IG02        trace.to_Tomcat02.row_count     2021-03-06T17:45:00+0800            3.000000   17.100000   3.000000  1.000000                 30
6   Tomcat02      trace.from_IG02.duration_mean     2021-03-06T17:45:00+0800            0.098667    0.095127   0.019477  1.037215                 30
7   Tomcat02           trace.self.duration_mean     2021-03-06T17:45:00+0800            0.014111    0.013836   0.003752  1.019872                 30
8   Tomcat02            trace.self.duration_p95     2021-03-06T17:45:00+0800            0.087750    0.082080   0.010725  1.069079                 30
9   Tomcat02               trace.self.row_count     2021-03-06T17:45:00+0800           36.000000  220.850000  42.000000  1.166667                 30
10      IG02               trace.self.row_count     2021-03-06T17:45:00+0800           32.000000  125.100000  32.900000  1.028125                 30
11      IG02    trace.to_Tomcat02.duration_mean     2021-03-06T17:45:00+0800            0.098667    0.095127   0.019477  1.037215                 30
12      MG01  trace.from_dockerB2.duration_mean     2021-03-06T17:45:00+0800            0.006000    0.060157   0.006000  1.000000                 29
13      MG01    trace.to_dockerA1.duration_mean     2021-03-06T17:46:00+0800            0.019500    0.074680   0.023657  1.213200                 29
14      MG01      trace.from_Tomcat03.row_count     2021-03-06T17:46:00+0800            3.000000   16.550000   3.000000  1.000000                 30
15      IG02        trace.to_Tomcat03.row_count     2021-03-06T17:46:00+0800            2.000000   16.550000   2.000000  1.000000                 30
16      MG01     trace.to_dockerA1.duration_p95     2021-03-06T17:46:00+0800            0.021700    0.218870   0.032100  1.479263                 29
17      IG02     trace.to_Tomcat04.duration_p95     2021-03-06T17:46:00+0800            0.040600    0.527425   0.041230  1.015517                 30
18      MG01      trace.from_dockerA2.row_count     2021-03-06T17:46:00+0800            1.000000    5.300000   1.000000  1.000000                 19
19      MG01      trace.from_dockerA1.row_count     2021-03-06T17:47:00+0800            4.000000    4.000000   1.000000  1.000000                 15)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse existing dataframes and tz from kernel state
try:
    metric_df
    trace_df
    log_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ["Tomcat02","MG01","IG02","Mysql02","apache02","Tomcat03","Tomcat04"]

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value']  = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value']    = pd.to_numeric(log_df['value'], errors='coerce')

# Window bounds (UTC+8)
window_start = pd.Timestamp('2021-03-06 17:45:00', tz=tz)
window_end   = pd.Timestamp('2021-03-06 18:00:00', tz=tz)

# Helper to add tz-localized timestamp column
def ensure_ts_local(df):
    if 'ts_local' not in df.columns:
        if 'timestamp' in df.columns:
            df['ts_local'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
        else:
            df['ts_local'] = pd.Series(pd.to_datetime([], unit='s')).dt.tz_localize(tz)
    return df

metric_df = ensure_ts_local(metric_df)
trace_df  = ensure_ts_local(trace_df)
log_df    = ensure_ts_local(log_df)

# 1) Compute global thresholds (P95,P5) and counts for each source/type before any filtering
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

log_thresholds = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# 2) Filter each file to the 15-min window and candidate components
metric_window_mask = metric_df['cmdb_id'].isin(candidates) & (metric_df['ts_local'] >= window_start) & (metric_df['ts_local'] <= window_end)
trace_window_mask  = trace_df['cmdb_id'].isin(candidates)  & (trace_df['ts_local']  >= window_start) & (trace_df['ts_local']  <= window_end)
log_window_mask    = log_df['cmdb_id'].isin(candidates)    & (log_df['ts_local']    >= window_start) & (log_df['ts_local']    <= window_end)

metric_window = metric_df[metric_window_mask].copy()
trace_window  = trace_df[trace_window_mask].copy()
log_window    = log_df[log_window_mask].copy()

# 3) Identify anomalies by merging thresholds and checking value >= p95 or <= p5
def compute_severity_val(v, p95, p5):
    # robust severity calculation
    if pd.isna(v):
        return np.nan
    # Prefer high-case if p95 exists and v >= p95
    if not pd.isna(p95) and v >= p95:
        if p95 == 0:
            return np.inf
        return v / p95
    # else low-case if p5 exists and v <= p5
    if not pd.isna(p5) and v <= p5:
        if v == 0:
            return np.inf if p5 == 0 else p5 / 1e-12
        return p5 / v
    return np.nan

# Metric anomalies
if not metric_window.empty:
    mw = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    mw = mw.dropna(subset=['value'])
    cond_m_high = mw['value'] >= mw['global_p95']
    cond_m_low  = mw['value'] <= mw['global_p5']
    mw_anom = mw[cond_m_high | cond_m_low].copy()
    if not mw_anom.empty:
        mw_anom['source_type'] = 'metric'
        mw_anom['name'] = mw_anom['kpi_name']
        mw_anom['severity'] = mw_anom.apply(lambda r: compute_severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
        # keep relevant columns
        mw_anom = mw_anom[['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series']]
    else:
        mw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series'])
else:
    mw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series'])

# Trace anomalies
if not trace_window.empty:
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    tw = tw.dropna(subset=['value'])
    cond_t_high = tw['value'] >= tw['global_p95']
    cond_t_low  = tw['value'] <= tw['global_p5']
    tw_anom = tw[cond_t_high | cond_t_low].copy()
    if not tw_anom.empty:
        tw_anom['source_type'] = 'trace'
        tw_anom['name'] = tw_anom['trace_name']
        tw_anom['severity'] = tw_anom.apply(lambda r: compute_severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
        tw_anom = tw_anom[['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series']]
    else:
        tw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series'])
else:
    tw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series'])

# Log anomalies
if not log_window.empty:
    lw = log_window.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    lw = lw.dropna(subset=['value'])
    cond_l_high = lw['value'] >= lw['global_p95']
    cond_l_low  = lw['value'] <= lw['global_p5']
    lw_anom = lw[cond_l_high | cond_l_low].copy()
    if not lw_anom.empty:
        lw_anom['source_type'] = 'log'
        lw_anom['name'] = lw_anom['log_name']
        lw_anom['severity'] = lw_anom.apply(lambda r: compute_severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
        lw_anom = lw_anom[['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series']]
    else:
        lw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series'])
else:
    lw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','value','global_p95','global_p5','severity','count_full_series'])

# Concatenate all anomalies
all_anom = pd.concat([mw_anom, tw_anom, lw_anom], ignore_index=True, sort=False)

# Prepare per-component summary
rows = []
for comp in candidates:
    comp_anom = all_anom[all_anom['cmdb_id'] == comp].copy()
    if comp_anom.empty:
        rows.append({
            'component': comp,
            'anomalies_count': 0,
            'earliest_anomaly_timestamp_in_window': None,
            'max_severity': None,
            'top_3_examples': []
        })
    else:
        # count distinct (source_type,name)
        distinct_count = comp_anom[['source_type','name']].drop_duplicates().shape[0]
        # earliest timestamp
        earliest_ts = comp_anom['ts_local'].min()
        earliest_iso = earliest_ts.strftime('%Y-%m-%dT%H:%M:%S%z')
        # max severity (ignore NaN)
        max_sev = comp_anom['severity'].replace([np.inf, -np.inf], np.inf).max()
        # top 3 examples by severity (take distinct name+source combination, pick max severity per name)
        comp_anom['name_key'] = comp_anom['source_type'] + '||' + comp_anom['name']
        # for each name_key get max severity and earliest ts for tie-break
        grouped_examples = (
            comp_anom.groupby('name_key').agg(
                source_type=('source_type','first'),
                name=('name','first'),
                max_severity=('severity','max'),
                earliest_ts=('ts_local','min')
            ).reset_index()
        )
        grouped_examples = grouped_examples.sort_values(['max_severity','earliest_ts'], ascending=[False,True]).head(3)
        top_examples = [(row.source_type, row.name, float(row.max_severity) if not pd.isna(row.max_severity) else None) for row in grouped_examples.itertuples(index=False)]
        rows.append({
            'component': comp,
            'anomalies_count': int(distinct_count),
            'earliest_anomaly_timestamp_in_window': earliest_iso,
            'max_severity': float(max_sev) if not pd.isna(max_sev) else None,
            'top_3_examples': top_examples
        })

summary_df = pd.DataFrame(rows)

# Label
summary_label = "candidate-component anomaly summary (15-min window)"

# Display compact result (one row per component, limited to candidates)
summary_label, summary_df.head(20)

```
Out[7]:
```
candidate-component anomaly summary (15-min window) — plain English:

- Tomcat02
  - anomalies_count: 127 distinct metric/trace/log names flagged in-window
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite (due to zero-threshold artifact)
  - top examples: log.error_count (log, inf), various OSLinux / Tomcat metrics (metric, high severity), trace row_count/duration entries (trace)

- MG01
  - anomalies_count: 124
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite
  - top examples: local-disk OSLinux metrics (metric, high severity), several trace and metric names

- IG02
  - anomalies_count: 102
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite
  - top examples: local-disk OSLinux metrics and MySQL-related metrics (metric, high severity), traces

- Mysql02
  - anomalies_count: 220
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite
  - top examples: many Mysql-MySQL_* metrics (metric, inf/high severity), OSLinux disk/process metrics

- apache02
  - anomalies_count: 80
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite
  - top examples: log.error_count (log, inf), OSLinux metrics (metric)

- Tomcat03
  - anomalies_count: 121
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite
  - top examples: log.error_count (log, inf), OSLinux metrics (metric)

- Tomcat04
  - anomalies_count: 119
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: infinite
  - top examples: log.error_count (log, inf), OSLinux metrics (metric)

Interpretation and caveats:
- All components show anomalies starting at 17:45 (UTC+8) within the 15-minute window.
- Many max_severity values are infinite because global P5/P95 thresholds (or values) are zero, causing division-by-zero artifacts — treat infinite severity as a strong signal but verify whether zeros reflect true drops or collection/aggregation issues.
- Common patterns: (a) metric drops to zero (especially Mysql02 and local-disk/process KPIs) and (b) log.error_count showing zero thresholds producing inf severity; traces also show row_count/duration deviations for Tomcat02/MG01/IG02.
- Actionable next steps: focus on Tomcat02 / MG01 / IG02 trace anomalies and validate metric collection and MySQL/service availability for Mysql02 and disk metrics; correlate with log volume drops for apache/Tomcat nodes.

The original code execution output of IPython Kernel is also provided below for reference:

('candidate-component anomaly summary (15-min window)',   component  anomalies_count earliest_anomaly_timestamp_in_window  max_severity                                     top_3_examples
0  Tomcat02              127             2021-03-06T17:45:00+0800           inf  [(log, log.error_count, inf), (metric, OSLinux...
1      MG01              124             2021-03-06T17:45:00+0800           inf  [(metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-...
2      IG02              102             2021-03-06T17:45:00+0800           inf  [(metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-...
3   Mysql02              220             2021-03-06T17:45:00+0800           inf  [(metric, Mysql-MySQL_3306_Aborted Clients, in...
4  apache02               80             2021-03-06T17:45:00+0800           inf  [(log, log.error_count, inf), (metric, OSLinux...
5  Tomcat03              121             2021-03-06T17:45:00+0800           inf  [(log, log.error_count, inf), (metric, OSLinux...
6  Tomcat04              119             2021-03-06T17:45:00+0800           inf  [(log, log.error_count, inf), (metric, OSLinux...)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse trace_df and tz if present
try:
    trace_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric and tz-localized timestamp
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
if 'ts_local' not in trace_df.columns:
    trace_df['ts_local'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)

# 1) Global thresholds (P95, P5) per (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# 2) Filter to 15-min window and selected cmdb_ids
window_start = pd.Timestamp('2021-03-06 17:45:00', tz=tz)
window_end   = pd.Timestamp('2021-03-06 18:00:00', tz=tz)
selected = ["Tomcat02","MG01","IG02"]

mask_window = (trace_df['ts_local'] >= window_start) & (trace_df['ts_local'] <= window_end) & (trace_df['cmdb_id'].isin(selected))
trace_window = trace_df[mask_window].copy()

# 3) Identify anomalous rows by merging thresholds
if trace_window.empty:
    label = "trace downstream-target anomaly counts (Tomcat02/MG01/IG02, 15-min window)"
    result_df = pd.DataFrame([{'note':'No trace rows for selected components in the 15-min window; no anomalies found.'}])
else:
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    tw = tw.dropna(subset=['value'])
    cond_high = tw['value'] >= tw['global_p95']
    cond_low  = tw['value'] <= tw['global_p5']
    tw_anom = tw[cond_high | cond_low].copy()
    if tw_anom.empty:
        label = "trace downstream-target anomaly counts (Tomcat02/MG01/IG02, 15-min window)"
        result_df = pd.DataFrame([{'note':'No anomalies (value >= P95 or <= P5) found for selected components in the 15-min window.'}])
    else:
        # 4) Parse trace_name for 'trace.to_<Target>.' pattern
        # extract target after 'trace.to_' up to first '.' or end
        tw_anom['to_target'] = tw_anom['trace_name'].str.extract(r'^trace\.to_([^\.]+)', expand=False)
        # 3) compute severity for each anomalous row
        def sev(v, p95, p5):
            if pd.isna(v):
                return np.nan
            if not pd.isna(p95) and v >= p95:
                return np.inf if p95 == 0 else v / p95
            if not pd.isna(p5) and v <= p5:
                if v == 0:
                    return np.inf if p5 == 0 else p5 / 1e-12
                return p5 / v
            return np.nan
        tw_anom['severity'] = tw_anom.apply(lambda r: sev(r['value'], r['global_p95'], r['global_p5']), axis=1)
        # Keep only rows that have a 'to' target (per instruction aggregate by target service)
        tw_to = tw_anom[tw_anom['to_target'].notna()].copy()
        if tw_to.empty:
            label = "trace downstream-target anomaly counts (Tomcat02/MG01/IG02, 15-min window)"
            result_df = pd.DataFrame([{'note':'No downstream "to" target anomalies found for selected components in the window.'}])
        else:
            # 5) Aggregate anomalies by target service
            # For example_trace_names we need up to 3 distinct trace_name examples with severity
            def top_examples_for_group(df_group, top_n=3):
                # get max severity per trace_name, then sort
                ex = (df_group.groupby('trace_name', as_index=False)
                      .agg(max_severity=('severity','max'),
                           earliest_ts=('ts_local','min'))
                      .sort_values(['max_severity','earliest_ts'], ascending=[False,True])
                      .head(top_n))
                # convert to list of tuples (trace_name, severity)
                return [(r.trace_name, float(r.max_severity) if not pd.isna(r.max_severity) else None) for r in ex.itertuples(index=False)]

            agg = (tw_to.groupby('to_target')
                   .agg(
                       anomalies_count = ('to_target','count'),
                       earliest_anomaly_timestamp = ('ts_local','min'),
                       max_severity = ('severity','max')
                   )
                   .reset_index()
                  )

            # add example trace names
            examples = []
            for tgt in agg['to_target']:
                grp = tw_to[tw_to['to_target'] == tgt]
                ex = top_examples_for_group(grp, top_n=3)
                examples.append(ex)
            agg['example_trace_names'] = examples
            # format earliest timestamp to ISO string with offset
            agg['earliest_anomaly_timestamp_in_window'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
            agg = agg.rename(columns={'to_target':'target_service'}).loc[:, ['target_service','anomalies_count','earliest_anomaly_timestamp_in_window','max_severity','example_trace_names']]
            # 6) Sort by anomalies_count desc then max_severity desc, limit top 20
            result_df = agg.sort_values(['anomalies_count','max_severity'], ascending=[False, False]).head(20).reset_index(drop=True)
            label = "trace downstream-target anomaly counts (Tomcat02/MG01/IG02, 15-min window)"

# Display label and compact aggregated result (top 20)
label, result_df

```
Out[8]:
```
Label: trace downstream-target anomaly counts (Tomcat02/MG01/IG02, 15-min window)

Summary (top downstream targets receiving anomalous trace signals from Tomcat02 / MG01 / IG02 between 2021-03-06 17:45–18:00 UTC+8):

- Tomcat02
  - anomalies_count: 11
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: ~1.62
  - examples: trace.to_Tomcat02.duration_p95 (severity ~1.6225), etc.

- dockerA1
  - anomalies_count: 10
  - earliest anomaly: 2021-03-06T17:46:00+08:00
  - max_severity: ~1.48
  - examples: trace.to_dockerA1.duration_p95 (severity ~1.4793), etc.

- Tomcat04
  - anomalies_count: 9
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: ~2.45
  - examples: trace.to_Tomcat04.duration_p95 (severity ~2.4542), etc.

- dockerB2
  - anomalies_count: 8
  - earliest anomaly: 2021-03-06T17:48:00+08:00
  - max_severity: ~5.18 (highest in list)
  - examples: trace.to_dockerB2.duration_p95 (severity ~5.1788), etc.

- dockerB1
  - anomalies_count: 8
  - earliest anomaly: 2021-03-06T17:49:00+08:00
  - max_severity: ~4.93
  - examples: trace.to_dockerB1.duration_p95 (severity ~4.9281), etc.

- MG01
  - anomalies_count: 8
  - earliest anomaly: 2021-03-06T17:50:00+08:00
  - max_severity: ~2.31
  - examples: trace.to_MG01.duration_mean (severity ~2.3149), etc.

- Tomcat03
  - anomalies_count: 6
  - earliest anomaly: 2021-03-06T17:46:00+08:00
  - max_severity: ~2.54
  - examples: trace.to_Tomcat03.duration_p95 (severity ~2.5373), etc.

- dockerA2, MG02, Tomcat01
  - anomalies_count: 6, 6, 5 respectively
  - earliest anomalies between 17:49–17:50; max severities in the ~1.77–2.03 range
  - examples: trace.to_dockerA2.row_count, trace.to_MG02.duration_p95, trace.to_Tomcat01.duration_p95

Interpretation and recommended next steps:
- The selected sources (Tomcat02, MG01, IG02) produced anomalous downstream traces targeting several services; anomalies begin as early as 17:45 and include moderate-to-high severity duration spikes (notably dockerB2 and dockerB1 had the largest severity ratios).
- Prioritize investigating downstream services with both high anomaly counts and high max severity (dockerB2, dockerB1, Tomcat04, Tomcat03, Tomcat02, MG01).
- Actions: inspect detailed traces and logs for those downstream targets at the listed earliest times, check service health and network connectivity between Tomcat02/MG01/IG02 and these targets, and correlate with metric/log drops observed earlier.

The original code execution output of IPython Kernel is also provided below for reference:

('trace downstream-target anomaly counts (Tomcat02/MG01/IG02, 15-min window)',   target_service  anomalies_count earliest_anomaly_timestamp_in_window  max_severity                                example_trace_names
0       Tomcat02               11             2021-03-06T17:45:00+0800      1.622507  [(trace.to_Tomcat02.duration_p95, 1.6225071729...
1       dockerA1               10             2021-03-06T17:46:00+0800      1.479263  [(trace.to_dockerA1.duration_p95, 1.4792626728...
2       Tomcat04                9             2021-03-06T17:45:00+0800      2.454167  [(trace.to_Tomcat04.duration_p95, 2.4541666666...
3       dockerB2                8             2021-03-06T17:48:00+0800      5.178846  [(trace.to_dockerB2.duration_p95, 5.1788461538...
4       dockerB1                8             2021-03-06T17:49:00+0800      4.928125  [(trace.to_dockerB1.duration_p95, 4.9281250000...
5           MG01                8             2021-03-06T17:50:00+0800      2.314893  [(trace.to_MG01.duration_mean, 2.3148929799344...
6       Tomcat03                6             2021-03-06T17:46:00+0800      2.537333  [(trace.to_Tomcat03.duration_p95, 2.5373333333...
7       dockerA2                6             2021-03-06T17:50:00+0800      2.000000  [(trace.to_dockerA2.row_count, 2.0), (trace.to...
8           MG02                6             2021-03-06T17:49:00+0800      1.766645  [(trace.to_MG02.duration_p95, 1.76664469347396...
9       Tomcat01                5             2021-03-06T17:49:00+0800      2.027434  [(trace.to_Tomcat01.duration_p95, 2.0274336283...)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present; otherwise load
try:
    metric_df
    trace_df
    log_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')

# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Candidate components and window
candidates = ["Tomcat02","MG01","IG02","Mysql02","apache02","Tomcat03","Tomcat04"]
window_start = pd.Timestamp('2021-03-06 17:45:00', tz=tz)
window_end   = pd.Timestamp('2021-03-06 18:00:00', tz=tz)

# Ensure numeric values and tz-localized timestamps (reuse columns if exist)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value']  = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value']    = pd.to_numeric(log_df['value'], errors='coerce')

if 'ts_local' not in metric_df.columns:
    metric_df['ts_local'] = pd.to_datetime(metric_df['timestamp'], unit='s').dt.tz_localize(tz)
if 'ts_local' not in trace_df.columns:
    trace_df['ts_local'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(tz)
if 'ts_local' not in log_df.columns:
    log_df['ts_local'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize(tz)

# Rule: compute global thresholds (P95,P5) BEFORE filtering
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

log_thresholds = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# Filter to window and candidates
metric_window = metric_df[(metric_df['cmdb_id'].isin(candidates)) & (metric_df['ts_local'] >= window_start) & (metric_df['ts_local'] <= window_end)].copy()
trace_window  = trace_df[(trace_df['cmdb_id'].isin(candidates))  & (trace_df['ts_local'] >= window_start) & (trace_df['ts_local'] <= window_end)].copy()
log_window    = log_df[(log_df['cmdb_id'].isin(candidates))    & (log_df['ts_local'] >= window_start) & (log_df['ts_local'] <= window_end)].copy()

# Severity function
def compute_severity_val(v, p95, p5):
    if pd.isna(v):
        return np.nan
    if not pd.isna(p95) and v >= p95:
        if p95 == 0:
            return np.inf
        return v / p95
    if not pd.isna(p5) and v <= p5:
        if v == 0:
            return np.inf if p5 == 0 else p5 / 1e-12
        return p5 / v
    return np.nan

# Identify anomalies for each source type (merge thresholds first)
# Metrics
if not metric_window.empty:
    mw = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    mw = mw.dropna(subset=['value'])
    mw_cond = (mw['value'] >= mw['global_p95']) | (mw['value'] <= mw['global_p5'])
    mw_anom = mw[mw_cond].copy()
    if not mw_anom.empty:
        mw_anom['source_type'] = 'metric'
        mw_anom['name'] = mw_anom['kpi_name']
        mw_anom['severity'] = mw_anom.apply(lambda r: compute_severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
        mw_anom = mw_anom[['cmdb_id','source_type','name','ts_local','severity']]
    else:
        mw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','severity'])
else:
    mw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','severity'])

# Traces
if not trace_window.empty:
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    tw = tw.dropna(subset=['value'])
    tw_cond = (tw['value'] >= tw['global_p95']) | (tw['value'] <= tw['global_p5'])
    tw_anom = tw[tw_cond].copy()
    if not tw_anom.empty:
        tw_anom['source_type'] = 'trace'
        tw_anom['name'] = tw_anom['trace_name']
        tw_anom['severity'] = tw_anom.apply(lambda r: compute_severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
        tw_anom = tw_anom[['cmdb_id','source_type','name','ts_local','severity']]
    else:
        tw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','severity'])
else:
    tw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','severity'])

# Logs
if not log_window.empty:
    lw = log_window.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    lw = lw.dropna(subset=['value'])
    lw_cond = (lw['value'] >= lw['global_p95']) | (lw['value'] <= lw['global_p5'])
    lw_anom = lw[lw_cond].copy()
    if not lw_anom.empty:
        lw_anom['source_type'] = 'log'
        lw_anom['name'] = lw_anom['log_name']
        lw_anom['severity'] = lw_anom.apply(lambda r: compute_severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
        lw_anom = lw_anom[['cmdb_id','source_type','name','ts_local','severity']]
    else:
        lw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','severity'])
else:
    lw_anom = pd.DataFrame(columns=['cmdb_id','source_type','name','ts_local','severity'])

# Combine anomalies
all_anom = pd.concat([tw_anom, mw_anom, lw_anom], ignore_index=True, sort=False)

# Build per-component rollup
rows = []
for comp in candidates:
    comp_traces = tw_anom[tw_anom['cmdb_id'] == comp] if not tw_anom.empty else pd.DataFrame()
    comp_metrics = mw_anom[mw_anom['cmdb_id'] == comp] if not mw_anom.empty else pd.DataFrame()
    comp_logs    = lw_anom[lw_anom['cmdb_id'] == comp] if not lw_anom.empty else pd.DataFrame()

    trace_anom_count = int(comp_traces['name'].nunique()) if not comp_traces.empty else 0
    metric_anom_count = int(comp_metrics['name'].nunique()) if not comp_metrics.empty else 0
    log_anom_count = int(comp_logs['name'].nunique()) if not comp_logs.empty else 0

    comp_all = pd.concat([comp_traces, comp_metrics, comp_logs], ignore_index=True, sort=False)
    if comp_all.empty:
        rows.append({
            'component': comp,
            'trace_anom_count': 0,
            'metric_anom_count': 0,
            'log_anom_count': 0,
            'total_anom_count': 0,
            'earliest_anomaly_timestamp_in_window': None,
            'max_severity': None,
            'top_example': None
        })
    else:
        total_anom_count = int(trace_anom_count + metric_anom_count + log_anom_count)
        # earliest timestamp
        earliest_ts = comp_all['ts_local'].min()
        earliest_iso = earliest_ts.strftime('%Y-%m-%dT%H:%M:%S%z')
        # max severity
        max_sev = comp_all['severity'].replace([np.inf, -np.inf], np.inf).max()
        # top example: pick row with max severity, tie-break by earliest timestamp
        comp_all_sorted = comp_all.sort_values(['severity','ts_local'], ascending=[False,True])
        top = comp_all_sorted.iloc[0]
        top_example = (top['source_type'], top['name'], float(top['severity']) if not pd.isna(top['severity']) else None)

        rows.append({
            'component': comp,
            'trace_anom_count': trace_anom_count,
            'metric_anom_count': metric_anom_count,
            'log_anom_count': log_anom_count,
            'total_anom_count': total_anom_count,
            'earliest_anomaly_timestamp_in_window': earliest_iso,
            'max_severity': float(max_sev) if not pd.isna(max_sev) else None,
            'top_example': top_example
        })

summary_df = pd.DataFrame(rows)

# Label
summary_label = "component anomaly rollup (15-min window)"

# Present compact result (one row per component)
summary_label, summary_df.head(20)

```
Out[9]:
```
component anomaly rollup (15-min window) — concise summary (window 2021-03-06 17:45 → 18:00 UTC+8)

- Tomcat02
  - trace_anom_count: 15, metric_anom_count: 109, log_anom_count: 3, total_anom_count: 127
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf (division-by-zero artifact)
  - top_example: (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s..., inf)

- MG01
  - trace_anom_count: 37, metric_anom_count: 87, log_anom_count: 0, total_anom_count: 124
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf
  - top_example: (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s..., inf)

- IG02
  - trace_anom_count: 15, metric_anom_count: 87, log_anom_count: 0, total_anom_count: 102
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf
  - top_example: (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s..., inf)

- Mysql02
  - trace_anom_count: 0, metric_anom_count: 220, log_anom_count: 0, total_anom_count: 220
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf
  - top_example: (metric, Mysql-MySQL_3306_Aborted Clients, inf)

- apache02
  - trace_anom_count: 0, metric_anom_count: 78, log_anom_count: 2, total_anom_count: 80
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf
  - top_example: (metric, OSLinux-CPU_CPU_CPULoad, inf)

- Tomcat03
  - trace_anom_count: 15, metric_anom_count: 103, log_anom_count: 3, total_anom_count: 121
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf
  - top_example: (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s..., inf)

- Tomcat04
  - trace_anom_count: 15, metric_anom_count: 101, log_anom_count: 3, total_anom_count: 119
  - earliest anomaly: 2021-03-06T17:45:00+08:00
  - max_severity: inf
  - top_example: (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s..., inf)

Notes and interpretation
- All components show anomalies starting at 17:45 (UTC+8) in this 15-min window.
- Most anomalies are in metrics (large metric_anom_count), with traces contributing notably for Tomcat02, MG01, IG02, Tomcat03/04. Log anomalies are few.
- Many max_severity values are infinite because some global P5/P95 thresholds (or observed values) are zero, producing division-by-zero; treat infinite severity as a strong signal but verify whether zeros are real (service drop) or an instrumentation/collection artifact.
- Actionable priorities: investigate trace-level issues for Tomcat02 / MG01 / IG02 (latency/row_count patterns) and validate metric collection and MySQL/service health for Mysql02 and local-disk/process metrics (zeros). Correlate with log volume drops for apache/Tomcat nodes.

The original code execution output of IPython Kernel is also provided below for reference:

('component anomaly rollup (15-min window)',   component  trace_anom_count  metric_anom_count  log_anom_count  total_anom_count earliest_anomaly_timestamp_in_window  max_severity                                        top_example
0  Tomcat02                15                109               3               127             2021-03-06T17:45:00+0800           inf  (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...
1      MG01                37                 87               0               124             2021-03-06T17:45:00+0800           inf  (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...
2      IG02                15                 87               0               102             2021-03-06T17:45:00+0800           inf  (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...
3   Mysql02                 0                220               0               220             2021-03-06T17:45:00+0800           inf    (metric, Mysql-MySQL_3306_Aborted Clients, inf)
4  apache02                 0                 78               2                80             2021-03-06T17:45:00+0800           inf             (metric, OSLinux-CPU_CPU_CPULoad, inf)
5  Tomcat03                15                103               3               121             2021-03-06T17:45:00+0800           inf  (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...
6  Tomcat04                15                101               3               119             2021-03-06T17:45:00+0800           inf  (metric, OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse dataframes from prior state; if not present, load them (light)
try:
    metric_df
    trace_df
    log_df
    error_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

# Ensure timezone-aware timestamp columns (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        col = 'ts_local'
        if col not in df.columns:
            df[col] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

# Incident window requested (as originally asked earlier)
incident_window_requested = "2021-03-07 01:30:00 to 2021-03-07 02:00:00 UTC+8"
window_start_req = pd.Timestamp('2021-03-07 01:30:00', tz=tz)
window_end_req   = pd.Timestamp('2021-03-07 02:00:00', tz=tz)

# Telemetry coverage summary: rows_in_window and nearest_before_timestamp (max ts < window_start_req)
def coverage_summary(df, name):
    if 'ts_local' not in df.columns or df.empty:
        return name, 0, 'empty'
    in_window = df[(df['ts_local'] >= window_start_req) & (df['ts_local'] <= window_end_req)]
    rows_in_window = int(in_window.shape[0])
    before_mask = df['ts_local'] < window_start_req
    if before_mask.sum() == 0:
        nearest_before = 'empty'
    else:
        nearest_before = df.loc[before_mask, 'ts_local'].max().strftime('%Y-%m-%dT%H:%M:%S%z')
    return name, rows_in_window, nearest_before

telemetry_coverage = {}
for fname, df in [('metric.csv', metric_df), ('trace.csv', trace_df), ('log.csv', log_df), ('error_logs.csv', error_df)]:
    nfile, count_win, before_ts = coverage_summary(df, fname)
    telemetry_coverage[nfile] = f"rows_in_window={count_win}, nearest_before={before_ts}"

# Compute global thresholds (P95/P5) BEFORE filtering for each source (reuse prior thresholds if exist)
# Metric thresholds
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# Trace thresholds
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# Log thresholds
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_thresholds = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(count_full_series='count',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 = lambda x: x.quantile(0.05))
)

# Filter to the 15-min window before nearest-before timestamp used in analysis (we already used 17:45->18:00 UTC+8)
# But user requested final report using already-analyzed outputs: use the earlier 15-min window for evidence
evidence_window_start = pd.Timestamp('2021-03-06 17:45:00', tz=tz)
evidence_window_end   = pd.Timestamp('2021-03-06 18:00:00', tz=tz)

metric_w = metric_df[(metric_df['ts_local'] >= evidence_window_start) & (metric_df['ts_local'] <= evidence_window_end)].copy()
trace_w  = trace_df[(trace_df['ts_local'] >= evidence_window_start) & (trace_df['ts_local'] <= evidence_window_end)].copy()
log_w    = log_df[(log_df['ts_local'] >= evidence_window_start) & (log_df['ts_local'] <= evidence_window_end)].copy()

# Helper severity computation (consistent with earlier)
def severity_val(v, p95, p5):
    if pd.isna(v):
        return None
    if (not pd.isna(p95)) and (v >= p95):
        if p95 == 0:
            return float('inf')
        return float(v / p95)
    if (not pd.isna(p5)) and (v <= p5):
        if v == 0:
            return float('inf') if p5 == 0 else float(p5 / 1e-12)
        return float(p5 / v)
    return None

# Build anomaly rows for traces in candidate sources (Tomcat02/MG01/IG02) and pick representative trace anomalies around 17:45
candidates = ["Tomcat02","MG01","IG02","Mysql02","apache02","Tomcat03","Tomcat04"]

tw = trace_w.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
tw = tw[tw['cmdb_id'].isin(['Tomcat02','MG01','IG02'])].copy()
tw = tw.dropna(subset=['value'])
tw['is_anom'] = (tw['value'] >= tw['global_p95']) | (tw['value'] <= tw['global_p5'])
tw = tw[tw['is_anom']].copy()
tw['severity'] = tw.apply(lambda r: severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
# pick up to 4 representative trace anomalies sorted by earliest timestamp and severity
trace_examples = tw.sort_values(['ts_local','severity'], ascending=[True, False]).head(4)

# Metric anomalies at timestamp 18:00 for Mysql02/apache02/Tomcat03 etc (representative)
metric_w_merged = metric_w.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
metric_w_merged = metric_w_merged[metric_w_merged['cmdb_id'].isin(['Mysql02','apache02','Tomcat03','Tomcat04'])].copy()
metric_w_merged = metric_w_merged.dropna(subset=['value'])
metric_w_merged['is_anom'] = (metric_w_merged['value'] >= metric_w_merged['global_p95']) | (metric_w_merged['value'] <= metric_w_merged['global_p5'])
metric_w_merged['severity'] = metric_w_merged.apply(lambda r: severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
# choose representative metric anomalies near 18:00 (value drops)
metric_examples = metric_w_merged.sort_values(['ts_local','severity'], ascending=[True, False]).head(3)

# Log anomalies representative (apache02/Tomcat03/Tomcat04)
log_w_merged = log_w.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
log_w_merged = log_w_merged[log_w_merged['cmdb_id'].isin(['apache02','Tomcat03','Tomcat04','Mysql01'])].copy()
log_w_merged = log_w_merged.dropna(subset=['value'])
log_w_merged['is_anom'] = (log_w_merged['value'] >= log_w_merged['global_p95']) | (log_w_merged['value'] <= log_w_merged['global_p5'])
log_w_merged['severity'] = log_w_merged.apply(lambda r: severity_val(r['value'], r['global_p95'], r['global_p5']), axis=1)
log_examples = log_w_merged.sort_values(['ts_local','severity'], ascending=[True, False]).head(2)

# Build top_evidence list (up to 6 items) combining trace_examples then metric_examples then log_examples
evidence_items = []
def row_to_evidence_source_trace(r):
    return {
        'source_file': 'trace.csv',
        'component': r.get('cmdb_id', None),
        'name': r.get('trace_name', None),
        'timestamp_iso': r['ts_local'].strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notna(r.get('ts_local')) else None,
        'observed_value': float(r['value']) if pd.notna(r.get('value')) else None,
        'global_p95': float(r['global_p95']) if pd.notna(r.get('global_p95')) else None,
        'global_p5': float(r['global_p5']) if pd.notna(r.get('global_p5')) else None,
        'severity': float(r['severity']) if pd.notna(r.get('severity')) and np.isfinite(r['severity']) else (float('inf') if r.get('severity')==float('inf') else None)
    }

def row_to_evidence_source_metric(r):
    return {
        'source_file': 'metric.csv',
        'component': r.get('cmdb_id', None),
        'name': r.get('kpi_name', None),
        'timestamp_iso': r['ts_local'].strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notna(r.get('ts_local')) else None,
        'observed_value': float(r['value']) if pd.notna(r.get('value')) else None,
        'global_p95': float(r['global_p95']) if pd.notna(r.get('global_p95')) else None,
        'global_p5': float(r['global_p5']) if pd.notna(r.get('global_p5')) else None,
        'severity': float(r['severity']) if pd.notna(r.get('severity')) and np.isfinite(r['severity']) else (float('inf') if r.get('severity')==float('inf') else None)
    }

def row_to_evidence_source_log(r):
    return {
        'source_file': 'log.csv',
        'component': r.get('cmdb_id', None),
        'name': r.get('log_name', None),
        'timestamp_iso': r['ts_local'].strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notna(r.get('ts_local')) else None,
        'observed_value': float(r['value']) if pd.notna(r.get('value')) else None,
        'global_p95': float(r['global_p95']) if pd.notna(r.get('global_p95')) else None,
        'global_p5': float(r['global_p5']) if pd.notna(r.get('global_p5')) else None,
        'severity': float(r['severity']) if pd.notna(r.get('severity')) and np.isfinite(r['severity']) else (float('inf') if r.get('severity')==float('inf') else None)
    }

for _, r in trace_examples.iterrows():
    if len(evidence_items) >= 6: break
    evidence_items.append(row_to_evidence_source_trace(r))
for _, r in metric_examples.iterrows():
    if len(evidence_items) >= 6: break
    evidence_items.append(row_to_evidence_source_metric(r))
for _, r in log_examples.iterrows():
    if len(evidence_items) >= 6: break
    evidence_items.append(row_to_evidence_source_log(r))

# definitive conclusion: whether root cause for requested window (2021-03-07 01:30-02:00) can be determined
# From prior scans we found the requested incident window had no telemetry (rows_in_window=0 for all), so definitive conclusion = No
definitive_conclusion = {
    'can_determine_root_cause': 'No',
    'justification': "No telemetry present in the requested incident window (rows_in_window=0 for metric/trace/log/error_logs), so root cause cannot be determined for that exact window."
}

# Best-effort hypothesis: choose Tomcat02 as likely component of interest (based on earlier trace anomalies at 2021-03-06T17:45+08)
best_effort_hypothesis = {
    'component': 'Tomcat02',
    'occurrence_datetime': '2021-03-06T17:45:00+08:00',
    'reason': 'Trace-level latency and row_count deviations involving Tomcat02 ↔ MG01/IG02 observed just before the requested window',
    'confidence': 'Medium',
    'caveats': [
        'Requested window had no telemetry; inference uses nearest-before data',
        'Several metric severities are infinite due to zero thresholds—possible collection artifact',
        'Cannot rule out monitoring/collection failures or traffic drops as alternate causes'
    ]
}

# Recommended next steps (ordered list of 5)
recommended_next_steps = [
    "Fetch raw traces for Tomcat02→MG01/IG02 at 2021-03-06T17:45–17:50 (nearest-before) to inspect spans and root cause.",
    "Confirm monitoring agent/collector health on Mysql02 and hosts showing metric drops (check agent logs and restart timestamps).",
    "Validate network connectivity and latency between Tomcat02, MG01, and IG02 around 17:45 (router/firewall metrics, cloud VPC metrics).",
    "Check application logs and request rates for Tomcat02, apache02, Tomcat03/04 at 17:45–18:00 to determine traffic drops versus collection gaps.",
    "If available, retrieve telemetry archives for the requested window (2021-03-07 01:30–02:00) from long-term storage/backup to confirm absence or recover data."
]

# Assemble final report dict (JSON-like)
final_best_effort_report = {
    'incident_window_requested': incident_window_requested,
    'telemetry_coverage_summary': telemetry_coverage,
    'definitive_conclusion': definitive_conclusion,
    'best_effort_hypothesis': best_effort_hypothesis,
    'top_evidence': evidence_items,
    'recommended_next_steps': recommended_next_steps
}

# Display compact report as the variable (text-like dict)
final_best_effort_report

```
Out[10]:
```
summary_text = (
    "Final best-effort report (concise):\n\n"
    "Incident window requested:\n"
    "2021-03-07 01:30:00 to 2021-03-07 02:00:00 UTC+8\n\n"
    "Telemetry coverage summary:\n"
    "- metric.csv: rows_in_window=0, nearest_before=2021-03-06T18:00:00+08:00\n"
    "- trace.csv:  rows_in_window=0, nearest_before=2021-03-06T17:59:00+08:00\n"
    "- log.csv:    rows_in_window=0, nearest_before=2021-03-06T18:00:00+08:00\n"
    "- error_logs.csv: rows_in_window=0, nearest_before=empty\n\n"
    "Definitive conclusion:\n"
    "No — the root cause for the requested window cannot be determined because there is no telemetry in that exact window (all files show 0 rows in-window).\n\n"
    "Best-effort hypothesis (nearest-before inference):\n"
    "- component: Tomcat02\n"
    "- occurrence_datetime: 2021-03-06T17:45:00+08:00\n"
    "- reason: trace-level latency and row_count deviations involving Tomcat02 ↔ MG01/IG02 observed immediately before the missing window\n"
    "- confidence: Medium\n"
    "- caveats: ['Using nearest-before data, not the requested window', 'Several metric severities are infinite due to zero thresholds (possible collection artifact)', 'Cannot rule out monitoring/collection failures or traffic drops']\n\n"
    "Top evidence (representative items from nearest-before window):\n"
    "1) trace.csv | Tomcat02 | trace.self.row_count | 2021-03-06T17:45:00+08:00 | value=36.0 | p95=220.85 | p5=42.0 | severity≈1.17\n"
    "2) trace.csv | MG01    | trace.from_dockerB1.duration_p95 | 2021-03-06T17:45:00+08:00 | value=0.03065 | p95=0.02649 | p5=0.00628 | severity≈1.16\n"
    "3) trace.csv | MG01    | trace.from_Tomcat04.row_count | 2021-03-06T17:45:00+08:00 | value=3.0 | p95=15.0 | p5=3.4 | severity≈1.13\n"
    "4) trace.csv | Tomcat02 | trace.self.duration_p95 | 2021-03-06T17:45:00+08:00 | value=0.08775 | p95=0.08208 | p5=0.01073 | severity≈1.07\n"
    "5) metric.csv | Mysql02 | Mysql-MySQL_3306_Aborted Clients | 2021-03-06T17:45:00+08:00 | value=0.0 | p95=0.0 | p5=0.0 | severity=inf (zero-threshold)\n"
    "6) metric.csv | Mysql02 | Mysql-MySQL_3306_Aborted Connects | 2021-03-06T17:45:00+08:00 | value=0.0 | p95=0.0 | p5=0.0 | severity=inf (zero-threshold)\n\n"
    "Interpretation (brief):\n"
    "- Nearest-before data shows modest trace latency/row_count deviations centered on Tomcat02 → MG01/IG02 around 17:45, plus many metric drops to zero (notably Mysql02 and disk/process metrics) at ~18:00. Some severity values are infinite due to zero thresholds—treat these with caution.\n\n"
    "Recommended next steps (ordered):\n"
    "1) Fetch raw traces for Tomcat02→MG01/IG02 at 2021-03-06T17:45–17:50 to inspect spans and pinpoint latency causes.\n"
    "2) Confirm monitoring agent/collector health on Mysql02 and hosts showing metric drops (check agent logs and restart timestamps).\n"
    "3) Validate network connectivity and latency between Tomcat02, MG01, and IG02 around 17:45 (router/firewall/VPC metrics).\n"
    "4) Check application request rates and logs for Tomcat02, apache02, Tomcat03/04 at 17:45–18:00 to distinguish traffic loss from collection issues.\n"
    "5) If available, retrieve telemetry archives for the requested window (2021-03-07 01:30–02:00) from long-term storage to confirm absence or recover data.\n\n"
    "Note: This is a fallback inference based on nearest-before telemetry; it is not a definitive root-cause for the requested window.\n"
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

{'incident_window_requested': '2021-03-07 01:30:00 to 2021-03-07 02:00:00 UTC+8', 'telemetry_coverage_summary': {'metric.csv': 'rows_in_window=0, nearest_before=2021-03-06T18:00:00+0800', 'trace.csv': 'rows_in_window=0, nearest_before=2021-03-06T17:59:00+0800', 'log.csv': 'rows_in_window=0, nearest_before=2021-03-06T18:00:00+0800', 'error_logs.csv': 'rows_in_window=0, nearest_before=empty'}, 'definitive_conclusion': {'can_determine_root_cause': 'No', 'justification': 'No telemetry present in the requested incident window (rows_in_window=0 for metric/trace/log/error_logs), so root cause cannot be determined for that exact window.'}, 'best_effort_hypothesis': {'component': 'Tomcat02', 'occurrence_datetime': '2021-03-06T17:45:00+08:00', 'reason': 'Trace-level latency and row_count deviations involving Tomcat02 ↔ MG01/IG02 observed just before the requested window', 'confidence': 'Medium', 'caveats': ['Requested window had no telemetry; inference uses nearest-before data', 'Several metric severities are infinite due to zero thresholds—possible collection artifact', 'Cannot rule out monitoring/collection failures or traffic drops as alternate causes']}, 'top_evidence': [{'source_file': 'trace.csv', 'component': 'Tomcat02', 'name': 'trace.self.row_count', 'timestamp_iso': '2021-03-06T17:45:00+0800', 'observed_value': 36.0, 'global_p95': 220.84999999999997, 'global_p5': 42.0, 'severity': 1.1666666666666667}, {'source_file': 'trace.csv', 'component': 'MG01', 'name': 'trace.from_dockerB1.duration_p95', 'timestamp_iso': '2021-03-06T17:45:00+0800', 'observed_value': 0.03065, 'global_p95': 0.02648999999999998, 'global_p5': 0.00628, 'severity': 1.1570403926009825}, {'source_file': 'trace.csv', 'component': 'MG01', 'name': 'trace.from_Tomcat04.row_count', 'timestamp_iso': '2021-03-06T17:45:00+0800', 'observed_value': 3.0, 'global_p95': 15.0, 'global_p5': 3.4000000000000004, 'severity': 1.1333333333333335}, {'source_file': 'trace.csv', 'component': 'Tomcat02', 'name': 'trace.self.duration_p95', 'timestamp_iso': '2021-03-06T17:45:00+0800', 'observed_value': 0.08775, 'global_p95': 0.08207999999999996, 'global_p5': 0.010725, 'severity': 1.0690789473684215}, {'source_file': 'metric.csv', 'component': 'Mysql02', 'name': 'Mysql-MySQL_3306_Aborted Clients', 'timestamp_iso': '2021-03-06T17:45:00+0800', 'observed_value': 0.0, 'global_p95': 0.0, 'global_p5': 0.0, 'severity': inf}, {'source_file': 'metric.csv', 'component': 'Mysql02', 'name': 'Mysql-MySQL_3306_Aborted Connects', 'timestamp_iso': '2021-03-06T17:45:00+0800', 'observed_value': 0.0, 'global_p95': 0.0, 'global_p5': 0.0, 'severity': inf}], 'recommended_next_steps': ['Fetch raw traces for Tomcat02→MG01/IG02 at 2021-03-06T17:45–17:50 (nearest-before) to inspect spans and root cause.', 'Confirm monitoring agent/collector health on Mysql02 and hosts showing metric drops (check agent logs and restart timestamps).', 'Validate network connectivity and latency between Tomcat02, MG01, and IG02 around 17:45 (router/firewall metrics, cloud VPC metrics).', 'Check application logs and request rates for Tomcat02, apache02, Tomcat03/04 at 17:45–18:00 to determine traffic drops versus collection gaps.', 'If available, retrieve telemetry archives for the requested window (2021-03-07 01:30–02:00) from long-term storage/backup to confirm absence or recover data.']}```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse existing dataframes if present; otherwise load
try:
    metric_df, trace_df, log_df, error_df, tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure ts_local exists and is tz-aware for each dataframe
def ensure_ts_local(df):
    if df.empty:
        df['ts_local'] = pd.Series([], dtype='datetime64[ns]')
    else:
        if 'ts_local' not in df.columns:
            df['ts_local'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    return df

metric_df = ensure_ts_local(metric_df)
trace_df  = ensure_ts_local(trace_df)
log_df    = ensure_ts_local(log_df)
error_df  = ensure_ts_local(error_df)

# Define range in UTC+8
range_start = pd.Timestamp('2021-03-07 01:00:00', tz=tz)
range_end   = pd.Timestamp('2021-03-07 03:00:00', tz=tz)

def summarize_range(df, file_name, sample_limit=20):
    if df.empty or df['ts_local'].dropna().empty:
        return {
            'file_name': file_name,
            'count_rows_in_range': 0,
            'min_timestamp_in_range': None,
            'max_timestamp_in_range': None,
            'sample_cmdb_ids_in_range': []
        }
    # Create mask safely (non-tz-aware empty entries handled)
    ts = df['ts_local']
    mask = (ts >= range_start) & (ts <= range_end)
    sub = df[mask]
    count = int(sub.shape[0])
    if count == 0:
        return {
            'file_name': file_name,
            'count_rows_in_range': 0,
            'min_timestamp_in_range': None,
            'max_timestamp_in_range': None,
            'sample_cmdb_ids_in_range': []
        }
    min_ts = sub['ts_local'].min().isoformat()
    max_ts = sub['ts_local'].max().isoformat()
    cmdbs = sorted(sub['cmdb_id'].dropna().unique())[:sample_limit]
    return {
        'file_name': file_name,
        'count_rows_in_range': count,
        'min_timestamp_in_range': min_ts,
        'max_timestamp_in_range': max_ts,
        'sample_cmdb_ids_in_range': cmdbs
    }

rows = [
    summarize_range(metric_df, 'metric.csv'),
    summarize_range(trace_df, 'trace.csv'),
    summarize_range(log_df, 'log.csv'),
    summarize_range(error_df, 'error_logs.csv')
]

summary_df = pd.DataFrame(rows)
summary_df

```
Out[11]:
```
All four telemetry files have no data in the requested time range (2021-03-07 01:00:00 → 2021-03-07 03:00:00 UTC+8):

- metric.csv: 0 rows in range, no min/max timestamp, sample_cmdb_ids = []
- trace.csv: 0 rows in range, no min/max timestamp, sample_cmdb_ids = []
- log.csv: 0 rows in range, no min/max timestamp, sample_cmdb_ids = []
- error_logs.csv: 0 rows in range, no min/max timestamp, sample_cmdb_ids = []

Conclusion: there is no telemetry available in that 2-hour window across metrics, traces, logs, or error logs. If you need to investigate further, I can (a) show the nearest-before and nearest-after timestamps we found earlier, (b) search archive/backup data for that window, or (c run fallback analyses on the nearest-before data already scanned. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

file_name  count_rows_in_range min_timestamp_in_range max_timestamp_in_range sample_cmdb_ids_in_range
0      metric.csv                    0                   None                   None                       []
1       trace.csv                    0                   None                   None                       []
2         log.csv                    0                   None                   None                       []
3  error_logs.csv                    0                   None                   None                       []```
```